---
title: "Staggered vs collocated: grid convergence of Stokes drag"
subtitle: "The two peclet.flow velocity placements on the Zick–Homsy sphere array — both converge to the benchmark, but the staggered grid at second order and the collocated grid at first."
author: "Peclet"
date: "2026-07-04"
categories: [flow, accuracy, staggered, collocated, Stokes, verification]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/benchmarks/staggered-vs-collocated/index.ipynb){target="_blank"}
&nbsp;Runs on a free Colab CPU runtime — the first cell installs `peclet` from PyPI.

## What this measures

`peclet.flow` ships **two velocity placements** behind one identical Python API:

- `flow.Solver` — the **staggered** MAC grid (velocities on cell faces, pressure at
  centres). The default, and *the* flow solver: exact discrete projection.
- `flow.SolverColocated` — a **collocated** grid (velocities and pressure both at
  cell centres) closed with an ABC *approximate* projection.

They solve the same equations on the same geometry; they differ only in *where the
velocity unknowns live*. This page puts that choice under a microscope on a problem
with an external ground truth — creeping flow through a periodic simple-cubic sphere
array, whose Stokes drag **Zick & Homsy (1982)** computed semi-analytically. We
refine the mesh and ask, for each grid:

1. **Does it converge, and at what order?**
2. **To *what value* does it converge** — the benchmark, or something offset from it?
3. **What does it cost** (iterations, wall-clock) to get there?

The punchline is **(1)**: both grids converge to the *same* Zick–Homsy value, but at
**different orders** — the staggered grid at **second order**, the collocated grid at
**first**. So at any practical resolution the collocated drag carries a larger (but
entirely *reducible*) error. Getting this right needed resolutions up to $N=128$ on a
GPU; the coarse grids alone are genuinely ambiguous (see the note below).

## The drag factor

A body force $F$ drives Stokes flow (no inertia) through a periodic cell of side $L$
containing one sphere of radius $R$ at solid fraction $\phi=\tfrac{4}{3}\pi R^3/L^3$.
The mean (superficial) velocity $\langle u\rangle$ fixes the dimensionless **drag
factor** — the per-sphere drag normalised by the isolated Stokes drag
$6\pi\mu R\langle u\rangle$:

$$
K=\frac{F\,L^{3}}{6\pi\mu R\,\langle u\rangle},
$$ {#eq-K}

which Zick & Homsy tabulate versus $\phi$. At $\phi=0.125$ (radius a quarter of the
cell) their value is $K_\text{ZH}=4.292$ — the target both grids must hit.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (installs peclet from PyPI on Colab/Binder)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    sys.path.insert(0, _local)
elif importlib.util.find_spec("peclet") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet"], check=True)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from peclet import flow

plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
BLUE, RED = "#1f77b4", "#d62728"

PHI0  = 0.125                    # solid fraction (sphere radius = L/4)
K_ZH  = 4.292                    # Zick & Homsy (1982), simple cubic, at PHI0
CENTRE = [(0.5, 0.5, 0.5)]       # one sphere per period, cell-centred

## The characterization harness

The same driver runs both grids — the *only* thing that changes between the two
sweeps is the solver class passed in. It builds the sphere as a signed distance
field, drives Stokes flow to a steady state, and reports the drag factor.

Because the collocated grid's *approximate* projection leaves the mean velocity with
a small residual jitter (the staggered grid's *exact* projection does not), we detect
a moderate steady state and then report $\langle u\rangle$ **averaged over a tail of
steps** — a fair, reproducible number for both grids rather than a single jittering
sample.

In [ ]:
#| label: harness
def sphere_sdf(N, phi, centres):
    """SDF of a periodic sphere lattice on an N³ grid (negative inside a sphere)."""
    n = len(centres)
    R = (3 * phi / (4 * np.pi * n)) ** (1 / 3) * N
    g = np.arange(N) + 0.5
    X, Y, Z = np.meshgrid(g, g, g, indexing="ij")
    best = np.full((N, N, N), 1e30)
    for cx, cy, cz in centres:
        dx = X - cx * N; dx -= N * np.round(dx / N)
        dy = Y - cy * N; dy -= N * np.round(dy / N)
        dz = Z - cz * N; dz -= N * np.round(dz / N)
        best = np.minimum(best, np.sqrt(dx * dx + dy * dy + dz * dz) - R)
    return best, R

def drag(SolverCls, N, phi=PHI0, centres=CENTRE, mu=0.1, F=1e-3, dt=80.0,
         warm_tol=1e-6, tail=40, max_steps=1000):
    sdf, R = sphere_sdf(N, phi, centres)
    s = SolverCls(N, N, N)
    s.set_rho(1.0); s.set_mu(mu); s.set_dt(dt)
    s.set_body_force(F, 0, 0); s.set_advection(False)     # Stokes: inertia off
    s.set_velocity_solver_params(150)
    s.set_pressure_multigrid(True, levels=max(2, int(np.log2(N)) - 1))
    s.set_pressure_pcg(True, 200, 1e-8)
    s.set_solid(sdf, cutcell_pressure=True, pressure_coarse="rediscretized")

    prev, warm, um, pit, t0 = 0.0, None, [], [], time.time()
    for it in range(max_steps):
        s.step()
        um.append(float(s.get_u().mean()))            # rolling history of <u>
        pit.append(int(s.last_pressure_iterations()))
        if warm is None:                              # still relaxing to steady state
            if it % 10 == 9:
                if it > 10 and abs(um[-1] - prev) < warm_tol * (abs(um[-1]) + 1e-30):
                    warm = it                         # reached moderate steady state
                prev = um[-1]
        elif it - warm >= tail:                       # collected a full post-warm tail
            break

    umean = float(np.mean(um[-tail:]))                # tail-average damps the jitter
    K = F * N ** 3 / len(centres) / (6 * np.pi * mu * R * umean)
    return dict(N=N, K=K, k=mu * umean / F, steps=it + 1, warm=warm,
                pit=float(np.mean(pit[-tail:])), secs=time.time() - t0)

## Sweep both grids

In [ ]:
#| label: sweep
Ns = [16, 24, 32, 40]
grids = {"staggered": flow.Solver, "collocated": flow.SolverColocated}
data = {name: [drag(cls, N) for N in Ns] for name, cls in grids.items()}

for name, rows in data.items():
    print(f"{name:>11s}: " + "  ".join(
        f"N={r['N']}→K={r['K']:.4f}({100*(r['K']-K_ZH)/K_ZH:+.2f}%)" for r in rows))

## Push to finer grids (the part that needs a GPU)

The live sweep above stops at $N=40$ so it runs on a Colab CPU. But **the coarse grids
are genuinely ambiguous**: at $N\le40$ the collocated error looks like it might be
levelling off at ~1%, and you can't tell "slowly converging" from "stuck at a floor".
The following resolutions — up to $N=128$, with the collocated case run to a tight
steady state — were computed on an **RTX 5080 GPU** and are quoted here as data (the
*Reproduce this* section gives the exact GPU command). They are the numbers that
settle the question.

In [ ]:
#| label: hires
# Signed error (K - K_ZH)/K_ZH in %, computed on GPU (RTX 5080) at a tight steady state.
# (staggered, collocated) per resolution.
HIRES = {48: (-0.08, +0.68), 56: (-0.05, +0.53), 64: (-0.018, +0.598),
         96: (+0.009, +0.397), 128: (+0.013, +0.299)}

# Merge the live coarse sweep (computed above) with the GPU points into one series/grid.
err_live = {name: {r["N"]: 100 * (r["K"] - K_ZH) / K_ZH for r in rows}
            for name, rows in data.items()}
allN = sorted(set(Ns) | set(HIRES))
E = {"staggered": [], "collocated": []}
for N in allN:
    es = err_live["staggered"].get(N, HIRES.get(N, (None, None))[0])
    ec = err_live["collocated"].get(N, HIRES.get(N, (None, None))[1])
    E["staggered"].append(es); E["collocated"].append(ec)
allN = np.array(allN, float)
for k in E: E[k] = np.array(E[k], float)

## Observed order of accuracy

With the full range in hand, read the **local order** between successive resolutions,
$p=\log(|e_{i-1}|/|e_i|)\,/\,\log(N_i/N_{i-1})$. This asks the sharp question directly:
how fast does each grid's error shrink?

In [ ]:
#| label: order
print(f"{'N pair':>10} | {'staggered p':>12} | {'collocated p':>13}")
for i in range(1, len(allN)):
    ps = np.log(abs(E['staggered'][i-1]) / abs(E['staggered'][i])) / np.log(allN[i]/allN[i-1])
    pc = np.log(abs(E['collocated'][i-1]) / abs(E['collocated'][i])) / np.log(allN[i]/allN[i-1])
    print(f"{int(allN[i-1]):>4}->{int(allN[i]):<4} | {ps:>+12.2f} | {pc:>+13.2f}")
print("\nHigh-resolution trend (N=64->96->128):")
print("  staggered  -> at the ~0.01% solver-tolerance floor (converged)")
print(f"  collocated -> {E['collocated'][-3]:+.3f}%, {E['collocated'][-2]:+.3f}%, "
      f"{E['collocated'][-1]:+.3f}%  (local order p ≈ 1.0)")

## The result

In [ ]:
#| label: fig-convergence
#| fig-cap: "Left: signed relative error vs resolution — BOTH grids head to zero (both converge to Zick & Homsy). Right: |error| on log-log axes — the staggered error runs parallel to O(h²) down to its ~0.01% solver-tolerance floor, while the collocated error runs parallel to O(h¹). Same limit, one order apart. The ~1% collocated error at N≈32 is first-order boundary error, halving each time N doubles — not a fixed bias."
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4.1))
col = {"staggered": BLUE, "collocated": RED}
mk = {"staggered": "o", "collocated": "s"}

a0.axhline(0, color="0.4", ls="--", lw=1)
for name in E:
    a0.plot(allN, E[name], mk[name] + "-", color=col[name], label=name)
a0.set(xlabel="N (cells per period)", ylabel="(K − K$_{ZH}$) / K$_{ZH}$  [%]",
       title="signed error → 0 for both")
a0.legend(fontsize=9)

for name in E:
    a1.loglog(allN, np.abs(E[name]), mk[name] + "-", color=col[name], label=name)
a1.loglog(allN, np.abs(E["collocated"][2]) * (allN / allN[2]) ** -1.0, ":",
          color=RED, lw=1.2, label="O(h¹)")
a1.loglog(allN, np.abs(E["staggered"][0]) * (allN / allN[0]) ** -2.0, "k--",
          lw=1, label="O(h²)")
a1.set(xlabel="N", ylabel="|K − K$_{ZH}$| / K$_{ZH}$  [%]",
       title="collocated ∥ O(h), staggered ∥ O(h²)")
a1.legend(fontsize=8)
fig.tight_layout()
plt.show()

## What it means

- **Both grids converge to Zick & Homsy — at different orders.** The staggered grid is
  **second order** (local $p\approx2$, reaching the ~0.01% solver-tolerance floor by
  $N=64$). The collocated grid is **first order** (local $p\approx1.0$ once the coarse
  transient clears: $+0.60\%\to+0.40\%\to+0.30\%$ at $N=64/96/128$). It converges to the
  *same* value, just as $O(h)$ instead of $O(h^2)$ — extrapolating to ~0.15% at $N=256$.
- **So the "~1% collocated error" is first-order boundary error, not a floor.** It is
  real at practical resolution but **reducible**: double $N$ and it roughly halves. An
  earlier version of this page mis-read the coarse ($N\le40$) data as a resolution-
  independent bias — the finer grids show it is not.
- **Why the order gap.** On the **staggered** grid each velocity lives on the same face
  whose fluid-area fraction (openness) enforces mass conservation, and the projection is
  *exact* — the no-slip and the incompressibility constraint are imposed on one
  consistent representation of the wall, and the drag inherits second order. On the
  **collocated** grid the no-slip is reconstructed at the cell centre (an axis-by-axis
  cut-cell stencil) while the mass flux uses the face fractions a half-cell away, and the
  projection is only *approximate* — a half-cell geometric inconsistency at a curved
  boundary that is $O(h)$ and does not cancel, giving first-order drag. On a **flat,
  grid-aligned** wall the two representations coincide and both grids are pointwise
  exact.
- **The fix is known.** A Basilisk-style *embedded-boundary* treatment — one geometric
  description of the cut cell (volume + face fractions, boundary centroid and normal)
  driving *both* the viscous no-slip and the pressure projection, with the wall gradient
  reconstructed along the true normal — restores second order on a collocated grid. See
  the detailed analysis in the flow solver's `doc/collocated_first_order_analysis.md`.

The headline for a practitioner: **use the staggered `flow.Solver` for
permeability/drag** — it is second-order and lands on the benchmark at modest
resolution. The collocated variant is a *consistent* solver (it converges to the same
answer) but first-order at curved immersed walls, so it needs finer grids there.

## Adapt this yourself

- **Change the solid fraction.** Raise `PHI0` toward close packing ($\phi_\max=\pi/6$
  for simple cubic) and update `K_ZH` from the Zick & Homsy table (see the
  [Zick–Homsy example](../../examples/zick-homsy/index.qmd)); the order gap persists.
- **Extend the live sweep.** Append `48, 64` to `Ns` if you have the wall-clock (or a
  GPU build) and watch the collocated error keep falling at first order rather than
  plateauing.
- **Swap the quantity of interest.** The same head-to-head applies to any QoI with a
  ground truth — a Poiseuille profile, a Taylor–Green decay rate — to map *where* the
  two placements share an order and where the collocated grid drops one.

## Reproduce this

```bash
pip install peclet
quarto render benchmarks/staggered-vs-collocated/index.qmd --execute
# ...or against a local source build:
PECLET_LOCAL_BUILD=/path/to/suite/flow/build_mpi OMP_NUM_THREADS=6 \
  OMP_PROC_BIND=spread OMP_PLACES=threads \
  quarto render benchmarks/staggered-vs-collocated/index.qmd --execute
```

The `HIRES` points (N=48–128) were computed with the same `drag()` harness pointed at a
**CUDA build** of the solver (`PECLET_LOCAL_BUILD=/path/to/suite/flow/build_cuda`),
running the collocated case to a tight steady state (`warm_tol=1e-7`, up to 4000 steps —
the ABC projection never fully quiets, so it tail-averages). On the RTX 5080 the whole
N=64/96/128 pass is a few minutes; on CPU it is hours, which is why they are quoted as
data rather than recomputed on every render.

::: {.callout-note}
## Ground truth
Zick, A. A. & Homsy, G. M. (1982), *Stokes flow through periodic arrays of spheres*,
J. Fluid Mech. **115**, 13–26 — the semi-analytic drag factors this page converges
against.
:::